In [1]:
# Fix TLS verification for websockets in this notebook
import ssl, certifi
ssl_ctx = ssl.create_default_context(cafile=certifi.where())
print("Using CA bundle:", certifi.where())


Using CA bundle: /Users/jonasbjaerke/Projects/data_analysis_practice/.venv/lib/python3.13/site-packages/certifi/cacert.pem


In [2]:
# Jupyter notebook cell
import asyncio, json, ssl, certifi
from datetime import datetime
import websockets

# --- config ---
ENDPOINT = "wss://jetstream1.us-east.bsky.network/subscribe"
# Filter to these collections on the server side:
WANTED = ["app.bsky.feed.repost"]
SAVE_REPOSTS_ANY_LANG = True   # set False to skip reposts without langs
OUTFILE = "bluesky_english_posts.jsonl"

# TLS context (fixes CERTIFICATE_VERIFY_FAILED on some setups)
ssl_ctx = ssl.create_default_context(cafile=certifi.where())

def is_en(langs):
    if not langs: return False
    return any(str(x).lower().startswith("en") for x in langs)

def is_target_commit(evt):
    if evt.get("kind") != "commit": 
        return False
    c = evt.get("commit") or {}
    if c.get("operation") != "create":
        return False
    return c.get("collection") in {"app.bsky.feed.post","app.bsky.feed.repost"}

def should_save(evt):
    c = evt["commit"]
    coll = c["collection"]
    rec = c.get("record") or {}
    if coll == "app.bsky.feed.post":
        return is_en(rec.get("langs"))
    if coll == "app.bsky.feed.repost":
        # Reposts typically have no text/langs. If you *must* filter by English,
        # you'd need to fetch the original post to inspect its langs.
        if is_en(rec.get("langs")):
            return True
        return SAVE_REPOSTS_ANY_LANG
    return False

def make_url():
    # multiple wantedCollections params are allowed
    parts = "&".join(f"wantedCollections={c}" for c in WANTED)
    return f"{ENDPOINT}?{parts}"

async def stream_and_save():
    url = make_url()
    print(f"[{datetime.utcnow().isoformat()}Z] Connecting to {url}")
    async with websockets.connect(url, ssl=ssl_ctx, max_size=None) as ws:
        print(f"[{datetime.utcnow().isoformat()}Z] Connected. Writing to {OUTFILE}")
        with open(OUTFILE, "a", encoding="utf-8") as f:
            async for message in ws:
                try:
                    evt = json.loads(message)
                except json.JSONDecodeError:
                    continue
                if is_target_commit(evt) and should_save(evt):
                    f.write(json.dumps(evt, ensure_ascii=False) + "\n")
                    f.flush()

# Run in a Jupyter cell:
await stream_and_save()


/var/folders/25/kgqdl4r17xv6mxdzwnzg5xsh0000gn/T/ipykernel_36165/570418239.py:49: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  print(f"[{datetime.utcnow().isoformat()}Z] Connecting to {url}")


[2025-10-20T21:59:35.318485Z] Connecting to wss://jetstream1.us-east.bsky.network/subscribe?wantedCollections=app.bsky.feed.repost


/var/folders/25/kgqdl4r17xv6mxdzwnzg5xsh0000gn/T/ipykernel_36165/570418239.py:51: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  print(f"[{datetime.utcnow().isoformat()}Z] Connected. Writing to {OUTFILE}")


[2025-10-20T21:59:35.733635Z] Connected. Writing to bluesky_english_posts.jsonl


CancelledError: 